# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets with their @id and name
print('Available Record Sets:')
for recset in metadata.record_sets:
    rec_id = recset.id
    name = getattr(recset, 'name', '(no name)')
    print(f"- @id: '{rec_id}' | name: '{name}'")
    # List the fields for each record set
    if hasattr(recset, 'fields'):
        print('  Fields:')
        for field in recset.fields:
            print(f"    - @id: '{field.id}' | name: '{getattr(field,'name','')}' | dataType: {getattr(field,'data_type','')}" )
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select record set(s) for extraction
# Example: get all their @id
record_sets = [recset.id for recset in metadata.record_sets]
print('Record Set @ids:', record_sets)
dataframes = {}

for record_set_id in record_sets:
    # Extract all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set '{record_set_id}'")
    else:
        print(f"No records found for record set '{record_set_id}'")

# Display columns for the first available data frame
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Columns for record set '{first_rs}':", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print('No record sets extracted.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# As an EDA example, select a record set with at least one numeric field
selected_rs = None
numeric_field_id = None

# Heuristically select the first record set with a float/integer field
for recset in metadata.record_sets:
    if hasattr(recset, 'fields'):
        numeric_fields = [field for field in recset.fields if getattr(field, 'data_type', None) in ['Float','Integer','Number']]
        if numeric_fields:
            selected_rs = recset.id
            numeric_field_id = numeric_fields[0].id
            print(f"Using record set '{selected_rs}' and numeric field '{numeric_field_id}' for EDA.")
            break

if selected_rs and numeric_field_id and selected_rs in dataframes:
    df = dataframes[selected_rs].copy()
    # Attempt conversion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field if possible
    group_field_id = None
    # Find a non-numeric field
    for field in recset.fields:
        if getattr(field, 'data_type', None) == 'Text' and field.id in df.columns:
            group_field_id = field.id
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print('No suitable numeric field found for EDA in available dataframes.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs and numeric_field_id and selected_rs in dataframes:
    df = dataframes[selected_rs]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.